# Generate Synthetic FLAIR v2 for Kaggle
Generate higher-quality synthetic FLAIR images on Kaggle by sampling multiple candidates per mask, scoring them, and saving only the best valid candidate to `synthetic_v2_kaggle`.

## Before You Run This Notebook
Attach these two Kaggle datasets to the notebook before running:

1. LGG dataset: a dataset containing the folder `lgg-mri-segmentation/`
2. DDPM checkpoints: a dataset containing one or more files named `checkpoint_v3_epoch_*.pt`

Expected behavior:

- the notebook searches under `/kaggle/input/**/lgg-mri-segmentation`
- the notebook searches under `/kaggle/input/**/checkpoint_v3_epoch_*.pt`
- if `checkpoint_v3_epoch_90.pt` exists, it uses that first
- otherwise it falls back to the highest available v3 epoch checkpoint

After generation finishes, the outputs are written to `/kaggle/working/outputs_v3/synthetic_v2_kaggle/` and zipped as `/kaggle/working/synthetic_v2_kaggle.zip`.

## Recommended Kaggle Dataset Layouts
These names are only examples. The notebook does not require exact dataset names, only the expected files inside `/kaggle/input`.

Example dataset 1: `lgg-mri-segmentation`
```text
/kaggle/input/lgg-mri-segmentation/
└── lgg-mri-segmentation/
    ├── TCGA_.../
    ├── TCGA_.../
    └── ...
```

Example dataset 2: `mask-to-mri-v3-checkpoints`
```text
/kaggle/input/mask-to-mri-v3-checkpoints/
├── checkpoint_v3_epoch_70.pt
├── checkpoint_v3_epoch_80.pt
└── checkpoint_v3_epoch_90.pt
```

Recommended Kaggle workflow:

1. Attach both datasets in the right-side Kaggle panel
2. Run the notebook normally
3. At the end, download `synthetic_v2_kaggle.zip` either from the clickable notebook link or from the Kaggle Output panel

## 1. Kaggle Setup & GPU Check

In [ ]:
import os
import torch

KAGGLE_INPUT = "/kaggle/input"
KAGGLE_WORKING = "/kaggle/working"
REPO_DIR = f"{KAGGLE_WORKING}/Mask-to-MRI"

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPUs available: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        gpu = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {gpu.name} ({gpu.total_memory/1e9:.1f} GB)")
    print(f"CUDA: {torch.version.cuda}")
else:
    print("No GPU detected")


## 2. Install Dependencies

In [ ]:
!pip install -q torch torchvision tifffile pillow matplotlib numpy tqdm albumentations scikit-image pyyaml
print("Dependencies installed")


## 3. Clone Repo

In [ ]:
import os

os.chdir(KAGGLE_WORKING)

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/AmineAitLaamim/Mask-to-MRI.git
    print("Repository cloned")
else:
    print("Repository already present")

os.chdir(REPO_DIR)
!git pull
print(f"Working dir: {os.getcwd()}")


## 4. Locate Dataset and Checkpoints

In [ ]:
import glob
from pathlib import Path

RAW_DIR = "/kaggle/input/datasets/amineee19/lgg-mri-segmentation/lgg-mri-segmentation"
if not os.path.exists(RAW_DIR):
    candidates = glob.glob(f"{KAGGLE_INPUT}/**/lgg-mri-segmentation", recursive=True)
    if candidates:
        RAW_DIR = max(candidates, key=len)
    else:
        raise FileNotFoundError("Could not find lgg-mri-segmentation under /kaggle/input")

    checkpoint_candidates = glob.glob(f"{KAGGLE_INPUT}/**/checkpoint_v3_epoch_*.pt", recursive=True)

    # Check specific user-provided path
    USER_SPECIFIC_CKPT = "/kaggle/input/datasets/amineee19/checkpoint-90/checkpoint_v3_epoch_90.pt"
    if os.path.exists(USER_SPECIFIC_CKPT):
        print(f"Using specific user checkpoint: {USER_SPECIFIC_CKPT}")
        checkpoint_candidates.insert(0, USER_SPECIFIC_CKPT)
    else:
        USER_CKPT_DIR = "/kaggle/input/datasets/amineee19/checkpoint-90"
        if os.path.exists(USER_CKPT_DIR):
            user_candidates = glob.glob(f"{USER_CKPT_DIR}/**/*.pt", recursive=True)
            if user_candidates:
                print(f"Found checkpoint in user path: {user_candidates[0]}")
                checkpoint_candidates.insert(0, user_candidates[0])

    if not checkpoint_candidates:
        raise FileNotFoundError(
            "No v3 checkpoints found under /kaggle/input. Upload a Kaggle dataset containing checkpoint_v3_epoch_*.pt files."
        )

    def _epoch_num(path: str) -> int:
        return int(Path(path).stem.split("_")[-1])

    epoch_90_candidates = [p for p in checkpoint_candidates if _epoch_num(p) == 90]
    if epoch_90_candidates:
        CKPT_PATH = epoch_90_candidates[0]
    else:
        CKPT_PATH = max(checkpoint_candidates, key=_epoch_num)

    print(f"RAW_DIR: {RAW_DIR}")
    print(f"Checkpoint: {CKPT_PATH}")


## 5. Create Output Directories

In [ ]:
OUTPUTS_DIR = f"{KAGGLE_WORKING}/outputs_v3"
OUTPUT_DIR = f"{OUTPUTS_DIR}/synthetic_v2_kaggle"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output dir: {OUTPUT_DIR}")


## 6. Import Modules & Load Model

In [ ]:
import sys
import json
import time

import numpy as np
import tifffile
from PIL import Image
from tqdm import tqdm

for _mod in [m for m in list(sys.modules.keys()) if m.startswith('src')]:
    del sys.modules[_mod]

sys.path.insert(0, REPO_DIR)

from src.med_ddpm_v3 import ConditionalDDPM, CONFIG
from src.dataset import get_patient_file_list, patient_level_split

CONFIG['raw_dir'] = RAW_DIR
CONFIG['synthetic_dir'] = OUTPUT_DIR

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

model = ConditionalDDPM(CONFIG).to(device)
ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)
if 'ema_state_dict' in ckpt:
    model.load_state_dict(ckpt['ema_state_dict'])
    print(f"Loaded EMA weights from epoch {ckpt['epoch']}")
else:
    model.load_state_dict(ckpt['model_state_dict'])
    print(f"Loaded raw model weights from epoch {ckpt['epoch']}")
model.eval()


## 7. Generation Settings

In [ ]:
DDIM_STEPS = 250
CFG_SCALE = 1.5
CANDIDATES_PER_MASK = 3
MAX_IMAGES = None  # set an int for a smaller run

MAX_TUMOR_RATIO = 0.08
MIN_TUMOR_PIXELS = 50

MIN_STD = 0.14
MAX_STD = 0.32
MAX_MEAN = -0.35
MIN_MEAN = -0.95
MIN_CONTRAST = 0.08

TARGET_MEAN = -0.62
TARGET_STD = 0.22

print(f"DDIM steps: {DDIM_STEPS}")
print(f"CFG scale: {CFG_SCALE}")
print(f"Candidates per mask: {CANDIDATES_PER_MASK}")
print(f"Max images: {MAX_IMAGES}")


## 8. Load Training Masks

In [ ]:
patient_data = get_patient_file_list(CONFIG['raw_dir'])
splits = patient_level_split(patient_data, seed=CONFIG['seed'])
train_pairs = splits['train']
print(f"Training pairs: {len(train_pairs)}")


## 9. Helper Functions

In [ ]:
def compute_quality_stats(fake_tensor, mask_tensor):
    fake_np = fake_tensor[0, 0].detach().cpu().numpy()
    mask_np = mask_tensor[0, 0].detach().cpu().numpy() > 0

    fake_mean = float(fake_np.mean())
    fake_std = float(fake_np.std())

    if mask_np.any():
        tumor_mean = float(fake_np[mask_np].mean())
    else:
        tumor_mean = fake_mean

    bg_mask = ~mask_np
    if bg_mask.any():
        bg_mean = float(fake_np[bg_mask].mean())
    else:
        bg_mean = fake_mean

    contrast = tumor_mean - bg_mean
    return {
        'mean': fake_mean,
        'std': fake_std,
        'tumor_mean': tumor_mean,
        'bg_mean': bg_mean,
        'contrast': float(contrast),
    }


def candidate_is_valid(stats):
    if stats['std'] < MIN_STD:
        return False, 'std too low'
    if stats['std'] > MAX_STD:
        return False, 'std too high'
    if stats['mean'] > MAX_MEAN:
        return False, 'mean too bright'
    if stats['mean'] < MIN_MEAN:
        return False, 'mean too dark'
    if stats['contrast'] < MIN_CONTRAST:
        return False, 'tumor contrast too weak'
    return True, 'ok'


def candidate_score(stats):
    mean_penalty = abs(stats['mean'] - TARGET_MEAN)
    std_penalty = abs(stats['std'] - TARGET_STD)
    contrast_bonus = stats['contrast']
    return contrast_bonus - 1.5 * mean_penalty - 1.0 * std_penalty


def to_png_uint8(fake_tensor):
    return ((fake_tensor[0, 0].detach().cpu().numpy() + 1.0) * 127.5).clip(0, 255).astype(np.uint8)


## 10. Generate Synthetic v2 on Kaggle

In [ ]:
existing = sorted(glob.glob(f"{OUTPUT_DIR}/synthetic_*.png"))
existing = [p for p in existing if not p.endswith('_mask.png')]
count = len(existing)
print(f"Existing synthetic_v2_kaggle images: {count}")

saved = 0
skipped_mask = 0
skipped_quality = 0
summary = []

for idx, (img_path, mask_path) in enumerate(tqdm(train_pairs, desc='Generating synthetic_v2_kaggle')):
    if MAX_IMAGES is not None and saved >= MAX_IMAGES:
        break

    m = tifffile.imread(mask_path)
    m = (np.squeeze(m) > 0).astype(np.uint8) * 255
    tumor_pixels = int((m > 0).sum())
    total_pixels = int(m.shape[0] * m.shape[1])
    tumor_ratio = tumor_pixels / total_pixels
    stem = Path(mask_path).stem.replace('_mask', '')

    if tumor_ratio > MAX_TUMOR_RATIO or tumor_pixels < MIN_TUMOR_PIXELS:
        print(f"Skipping {stem} - tumor ratio={tumor_ratio:.3f}, pixels={tumor_pixels}")
        skipped_mask += 1
        continue

    m_norm = (m.astype(np.float32) / 127.5) - 1.0
    mask_t = torch.from_numpy(m_norm).unsqueeze(0).unsqueeze(0).to(device)

    candidates = []
    for candidate_idx in range(CANDIDATES_PER_MASK):
        with torch.no_grad():
            fake = model.sample(mask_t, ddim_steps=DDIM_STEPS, cfg_scale=CFG_SCALE)

        stats = compute_quality_stats(fake, mask_t)
        valid, reason = candidate_is_valid(stats)
        stats['valid'] = valid
        stats['reason'] = reason
        stats['score'] = candidate_score(stats) if valid else -1e9
        stats['candidate_idx'] = candidate_idx
        stats['fake_tensor'] = fake.detach().cpu()
        candidates.append(stats)

    valid_candidates = [c for c in candidates if c['valid']]
    if not valid_candidates:
        best_attempt = max(candidates, key=lambda c: c['score'])
        print(
            f"Skipping {stem} - no valid candidate "
            f"(best attempt std={best_attempt['std']:.3f}, mean={best_attempt['mean']:.3f}, contrast={best_attempt['contrast']:.3f}, reason={best_attempt['reason']})"
        )
        skipped_quality += 1
        continue

    best = max(valid_candidates, key=lambda c: c['score'])
    count += 1
    saved += 1
    new_name = f"synthetic_{count:04d}"

    fake_png = to_png_uint8(best['fake_tensor'])
    Image.fromarray(fake_png, mode='L').save(os.path.join(OUTPUT_DIR, f"{new_name}.png"))
    Image.fromarray(m, mode='L').save(os.path.join(OUTPUT_DIR, f"{new_name}_mask.png"))

    record = {
        'name': new_name,
        'source_stem': stem,
        'tumor_ratio': tumor_ratio,
        'tumor_pixels': tumor_pixels,
        'mean': best['mean'],
        'std': best['std'],
        'contrast': best['contrast'],
        'score': best['score'],
        'candidate_idx': best['candidate_idx'],
    }
    summary.append(record)
    print(
        f"Saved {new_name} (was {stem}) - std={best['std']:.3f}, mean={best['mean']:.3f}, "
        f"contrast={best['contrast']:.3f}, score={best['score']:.3f}, tumor={tumor_ratio:.1%}"
    )

summary_path = os.path.join(OUTPUT_DIR, 'generation_summary.json')
with open(summary_path, 'w', encoding='utf-8') as f:
    json.dump({
        'saved': saved,
        'skipped_mask': skipped_mask,
        'skipped_quality': skipped_quality,
        'ddim_steps': DDIM_STEPS,
        'cfg_scale': CFG_SCALE,
        'candidates_per_mask': CANDIDATES_PER_MASK,
        'checkpoint_path': CKPT_PATH,
        'records': summary,
    }, f, indent=2)

print('\nGeneration complete')
print(f"Saved: {saved}")
print(f"Skipped by mask filter: {skipped_mask}")
print(f"Skipped by quality filter: {skipped_quality}")
print(f"Output folder: {OUTPUT_DIR}")
print(f"Summary: {summary_path}")


## 11. Preview Saved Results

In [ ]:
import matplotlib.pyplot as plt

synthetic_files = sorted(glob.glob(f"{OUTPUT_DIR}/synthetic_*.png"))
synthetic_files = [f for f in synthetic_files if not f.endswith('_mask.png')]
n = min(4, len(synthetic_files))

if n == 0:
    print('No saved synthetic_v2_kaggle images yet')
else:
    fig, axes = plt.subplots(n, 2, figsize=(8, 3 * n))
    if n == 1:
        axes = np.array([axes])

    for i in range(n):
        fake_path = synthetic_files[i]
        mask_path = fake_path.replace('.png', '_mask.png')
        fake_img = np.array(Image.open(fake_path))
        mask_img = np.array(Image.open(mask_path))

        axes[i, 0].imshow(mask_img, cmap='gray')
        axes[i, 0].set_title(Path(mask_path).name)
        axes[i, 0].axis('off')

        axes[i, 1].imshow(fake_img, cmap='gray')
        axes[i, 1].set_title(Path(fake_path).name)
        axes[i, 1].axis('off')

    plt.tight_layout()
    plt.show()


## 12. Package Output for Download or Reuse

In [ ]:
import shutil
from IPython.display import FileLink, display

archive_base = f"{KAGGLE_WORKING}/synthetic_v2_kaggle"
shutil.make_archive(archive_base, 'zip', OUTPUT_DIR)
print(f"Created archive: {archive_base}.zip")
print("You can download it from the Kaggle Output panel, or click the link below inside the notebook.")
display(FileLink(f"{archive_base}.zip"))
